# Домашнее задание по теме "Рекурентные сети"

## Задание 1. Обучить нейронную сеть решать шифр Цезаря

1. Написать алгоритм шифра Цезаря для генерации выборки (сдвиг на К каждой буквы. Например, при сдвиге на 2 буква “А” переходит в букву “В” и тп)
1. Сделать нейронную сеть
1. Обучить ее (вход - зашифрованная фраза, выход - дешифрованная фраза)
Проверить качество

### 1.1 Алгоритм шифра Цезаря

In [5]:
import pandas as pd
import time
import random

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split

В качестве корпуса слов будем использовать данные из датасета data.csv

In [6]:
df = pd.read_csv('../datas/data.csv')
df.head()

,Unnamed: 0,id,episode_id,number,raw_text,timestamp_in_ms,speaking_line,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
0,0,10368,35,29,"Lisa Simpson: Maggie, look. What's that?",235000,True,9,5.0,Lisa Simpson,Simpson Home,"Maggie, look. What's that?",maggie look whats that,4.0
1,1,10369,35,30,Lisa Simpson: Lee-mur. Lee-mur.,237000,True,9,5.0,Lisa Simpson,Simpson Home,Lee-mur. Lee-mur.,lee-mur lee-mur,2.0
2,2,10370,35,31,Lisa Simpson: Zee-boo. Zee-boo.,239000,True,9,5.0,Lisa Simpson,Simpson Home,Zee-boo. Zee-boo.,zee-boo zee-boo,2.0
3,3,10372,35,33,Lisa Simpson: I'm trying to teach Maggie that ...,245000,True,9,5.0,Lisa Simpson,Simpson Home,I'm trying to teach Maggie that nature doesn't...,im trying to teach maggie that nature doesnt e...,24.0
4,4,10374,35,35,"Lisa Simpson: It's like an ox, only it has a h...",254000,True,9,5.0,Lisa Simpson,Simpson Home,"It's like an ox, only it has a hump and a dewl...",its like an ox only it has a hump and a dewlap...,18.0


In [15]:
corpus = df['normalized_text'].tolist()
corpus[:10]

['maggie look whats that',
 'lee-mur lee-mur',
 'zee-boo zee-boo',
 'im trying to teach maggie that nature doesnt end with the barnyard i want her to have all the advantages that i didnt have',
 'its like an ox only it has a hump and a dewlap hump and dew-lap hump and dew-lap',
 'you know his blood type how romantic',
 'oh yeah whats my shoe size',
 'ring',
 'yes dad',
 'ooh look maggie what is that do-dec-ah-edron dodecahedron']

Функция шифрования

In [8]:
def caesar_cipher(text, shift, alphabet):
    """
    Универсальный шифр Цезаря.
    
    Parameters:
        text (str): Исходный текст
        shift (int): Величина сдвига
        alphabet (str): Строка со всеми буквами алфавита в нужном порядке
    
    Returns:
        str: Зашифрованный текст
    """
    # Создаем словари для маппинга
    char_to_index = {char: idx for idx, char in enumerate(alphabet)}
    index_to_char = {idx: char for idx, char in enumerate(alphabet)}
    
    # Для прописных букв (если алфавит содержит строчные)
    upper_alphabet = alphabet.upper()
    upper_char_to_index = {char: idx for idx, char in enumerate(upper_alphabet)}
    upper_index_to_char = {idx: char for idx, char in enumerate(upper_alphabet)}
    
    result = []
    
    for char in text:
        if char in char_to_index:
            new_index = (char_to_index[char] + shift) % len(alphabet)
            result.append(index_to_char[new_index])
        elif char in upper_char_to_index:
            new_index = (upper_char_to_index[char] + shift) % len(alphabet)
            result.append(upper_index_to_char[new_index])
        else:
            result.append(char)
    
    return ''.join(result)

Функции кодирования

In [9]:
def create_vocab(alphabet):
    """Создаёт словари кодирования"""
    char_to_idx = {char: idx for idx, char in enumerate(alphabet)}
    idx_to_char = {idx: char for idx, char in enumerate(alphabet)}
    return char_to_idx, idx_to_char, len(alphabet)


def encode_text(text, char_to_idx, vocab_size, max_seq_len):
    """Конвертирует текст в тензор индексов с паддингом"""
    default_idx = char_to_idx.get(' ', vocab_size - 1)
    indices = [char_to_idx.get(c, default_idx) for c in text[:max_seq_len]]
    
    while len(indices) < max_seq_len:
        indices.append(default_idx)
        
    return torch.tensor(indices, dtype=torch.long)


def decode_tensor(tensor, idx_to_char, vocab_size):
    """Конвертирует тензор обратно в текст"""
    if tensor.dim() == 2:
        tensor = tensor[0]
    
    chars = []
    for idx in tensor:
        idx_val = idx.item() if hasattr(idx, 'item') else int(idx)
        if idx_val < vocab_size:
            chars.append(idx_to_char.get(idx_val, ''))
    return ''.join(chars)

Функция генерации датасета

In [16]:
def generate_dataset(corpus, alphabet, max_seq_len=50, shift=3, test_size=0.2, random_state=42):
    """
    Генерирует X_train, X_test, y_train, y_test из корпуса текстов
    
    Parameters:
        corpus: список строк (текстов)
        alphabet: строка алфавита
        max_seq_len: максимальная длина последовательности
        shift: сдвиг Цезаря
        test_size: доля тестовой выборки
        random_state: seed для воспроизводимости
    
    Returns:
        X_train, X_test, y_train, y_test, char_to_idx, idx_to_char
    """
    
    # Создаём словари кодирования
    char_to_idx, idx_to_char, vocab_size = create_vocab(alphabet)
    
    X_encrypted = []
    y_original = []
    
    print(f"Processing {len(corpus)} texts...")
    
    for text in corpus:
        original = str(text).lower().strip()
        if len(original) == 0:
            continue
        
        # Шифруем с помощью функции caesar_cipher()
        encrypted = caesar_cipher(original, shift, alphabet)
        
        # Кодируем в тензоры
        X_encrypted.append(encode_text(encrypted, char_to_idx, vocab_size, max_seq_len))
        y_original.append(encode_text(original, char_to_idx, vocab_size, max_seq_len))
    
    # Объединяем в тензоры
    X = torch.stack(X_encrypted)
    y = torch.stack(y_original)
    
    # Разбиваем на train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    
    print(f"Dataset created:")
    print(f"  Total samples: {len(X)}")
    print(f"  Train: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
    print(f"  Test:  {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")
    print(f"  Sequence length: {max_seq_len}")
    print(f"  Vocab size: {vocab_size}")
    print(f"  Caesar shift: {shift}")
    
    return X_train, X_test, y_train, y_test, char_to_idx, idx_to_char

Разобъем датасет на обучающую и тестовую выборки

In [11]:
ENGLISH_ALPHABET = 'abcdefghijklmnopqrstuvwxyz '
MAX_SEQ_LEN = 50
SHIFT = 3

In [18]:
X_train, X_test, y_train, y_test, char_to_idx, idx_to_char = generate_dataset(
    corpus=corpus,
    alphabet=ENGLISH_ALPHABET,
    max_seq_len=MAX_SEQ_LEN,
    shift=SHIFT,
    test_size=0.2,
    random_state=42
)
    
# Проверяем
print(f"\n{'='*50}")
print("ПРОВЕРКА")
print(f"{'='*50}")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
    
# Декодируем пример
idx = 0
x_text = decode_tensor(X_train[idx], idx_to_char, len(ENGLISH_ALPHABET))
y_text = decode_tensor(y_train[idx], idx_to_char, len(ENGLISH_ALPHABET))
    
print(f"\nПример {idx}:")
print(f"  X (зашифровано): '{x_text}'")
print(f"  y (оригинал):    '{y_text}'")

Processing 11639 texts...
Dataset created:
  Total samples: 11639
  Train: 9311 (80.0%)
  Test:  2328 (20.0%)
  Sequence length: 50
  Vocab size: 27
  Caesar shift: 3

ПРОВЕРКА
X_train shape: torch.Size([9311, 50])
y_train shape: torch.Size([9311, 50])

Пример 0:
  X (зашифровано): 'vrclcjxhvvcwkhcohvvrqckhuhclv                     '
  y (оригинал):    'so i guess the lesson here is                     '


### 1.2 Нейронная сеть

Модель

In [51]:
class CaesarDecryptorRNN(nn.Module):
    def __init__(self, vocab_size, embeded_dim=30, hidden_dim=128):
        """
        RNN для дешифрования шифра Цезаря
        """
        super(CaesarDecryptorRNN, self).__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        # Слой эмбеддингов:
        self.embedding = nn.Embedding(vocab_size, embeded_dim)
        # Рекуррентный слой:
        self.rnn = nn.RNN(
            input_size=embeded_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )
        # Выходной линейный слой:
        self.out = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x, hidden=None):
        x = self.embedding(x)
        x, hidden = self.rnn(x, hidden)
        logits = self.out(x)
    
        return logits, hidden
    
    def predict(self, x):
        self.eval()
        with torch.no_grad():
            logits, _ = self.forward(x)
            predictions = logits.argmax(dim=-1)
        return predictions

In [52]:
def get_device():
    """Определяет доступное устройство: GPU если есть, иначе CPU"""
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print(f"GPU доступен: {torch.cuda.get_device_name(0)}")
    else:
        device = torch.device('cpu')
        print("GPU не доступен, используется CPU")
    return device


def train_epoch(model, X_train, y_train, criterion, optimizer, batch_size=32, device=None):
    if device is None:
        raise ValueError("device должен быть указан (torch.device('cpu') или torch.device('cuda'))")
    
    model.train()
    total_loss = 0
    n_batches = 0
    
    # Перемешиваем данные
    perm = torch.randperm(len(X_train))
    X_train = X_train[perm]
    y_train = y_train[perm]
    
    for i in range(0, len(X_train), batch_size):
        # Берём батч и переносим на устройство
        x_batch = X_train[i:i+batch_size].to(device)
        y_batch = y_train[i:i+batch_size].to(device)
        
        optimizer.zero_grad()
        
        logits, _ = model(x_batch)
        loss = criterion(logits.view(-1, model.vocab_size), y_batch.view(-1))
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        n_batches += 1
    
    return total_loss / n_batches


def evaluate(model, X_test, y_test, criterion, device=None):
    """
    Оценка на тестовой выборке
    """
    if device is None:
        raise ValueError("device должен быть указан")
    
    model.eval()
    
    X_test = X_test.to(device)
    y_test = y_test.to(device)
    
    with torch.no_grad():
        logits, _ = model(X_test)
        loss = criterion(logits.view(-1, model.vocab_size), y_test.view(-1))
        
        predictions = logits.argmax(dim=-1)
        correct = (predictions == y_test).sum().item()
        total = y_test.numel()
        
        accuracy = correct / total * 100
    
    return loss.item(), accuracy


def train_model(model, X_train, y_train, X_test, y_test, 
                epochs=50, batch_size=32, lr=0.001, device=None):
    """
    Полный цикл обучения с автоматическим выбором устройства
    """
    
    # Автоматический выбор устройства
    if device is None:
        device = get_device()
    else:
        device = torch.device(device)
    
    # Переносим модель и данные на устройство
    model = model.to(device)
    X_train = X_train.to(device)
    y_train = y_train.to(device)
    X_test = X_test.to(device)
    y_test = y_test.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    print(f"\nTraining on {device}")
    print(f"Epochs: {epochs}, Batch size: {batch_size}, LR: {lr}")
    print(f"{'='*50}")
    
    for epoch in range(epochs):
        # Передаём device явно
        train_loss = train_epoch(model, X_train, y_train, criterion, 
                                 optimizer, batch_size, device)
        test_loss, test_acc = evaluate(model, X_test, y_test, criterion, device)
        
        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch:3d} | Train Loss: {train_loss:.4f} | "
                  f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")
    
    return model, device

In [53]:
VOCAB_SIZE = len(ENGLISH_ALPHABET)

In [54]:
model = CaesarDecryptorRNN(
    vocab_size=VOCAB_SIZE,
    embeded_dim=30,
    hidden_dim=128
)

In [55]:
model

CaesarDecryptorRNN(
  (embedding): Embedding(27, 30)
  (rnn): RNN(30, 128, batch_first=True)
  (out): Linear(in_features=128, out_features=27, bias=True)
)

In [56]:
trained_model, device = train_model(
    model,
    X_train,
    y_train,
    X_test,
    y_test,
    epochs=100,
    batch_size=64,
    lr=0.002)

GPU доступен: NVIDIA GeForce RTX 3060

Training on cuda
Epochs: 100, Batch size: 64, LR: 0.002
Epoch   0 | Train Loss: 0.3764 | Test Loss: 0.0150 | Test Acc: 99.88%
Epoch  10 | Train Loss: 0.0006 | Test Loss: 0.0007 | Test Acc: 99.98%
Epoch  20 | Train Loss: 0.0002 | Test Loss: 0.0006 | Test Acc: 99.98%
Epoch  30 | Train Loss: 0.0002 | Test Loss: 0.0006 | Test Acc: 99.99%
Epoch  40 | Train Loss: 0.0002 | Test Loss: 0.0004 | Test Acc: 99.99%
Epoch  50 | Train Loss: 0.0002 | Test Loss: 0.0005 | Test Acc: 99.99%
Epoch  60 | Train Loss: 0.0000 | Test Loss: 0.0005 | Test Acc: 99.99%
Epoch  70 | Train Loss: 0.0000 | Test Loss: 0.0006 | Test Acc: 99.99%
Epoch  80 | Train Loss: 0.0000 | Test Loss: 0.0007 | Test Acc: 99.99%
Epoch  90 | Train Loss: 0.0000 | Test Loss: 0.0007 | Test Acc: 99.99%
Epoch  99 | Train Loss: 0.0000 | Test Loss: 0.0008 | Test Acc: 99.99%


In [62]:
# Сохранение
torch.save(model, 'caesar_model_full.pth')

# Загрузка
model = torch.load('caesar_model_full.pth', weights_only=False)
model.eval()

CaesarDecryptorRNN(
  (embedding): Embedding(27, 30)
  (rnn): RNN(30, 128, batch_first=True)
  (out): Linear(in_features=128, out_features=27, bias=True)
)

In [63]:
# Определяем устройство
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"Модель загружена на {device}")

Модель загружена на cuda


In [64]:
def decrypt_with_model(model, encrypted_text, char_to_idx, idx_to_char, 
                       max_seq_len=50, device='cpu'):
    """
    Дешифрует текст с помощью загруженной модели
    """
    model.eval()
    
    # Кодируем текст в индексы
    default_idx = char_to_idx.get(' ', len(char_to_idx) - 1)
    indices = [char_to_idx.get(c, default_idx) for c in encrypted_text[:max_seq_len]]
    
    # Паддинг
    while len(indices) < max_seq_len:
        indices.append(default_idx)
    
    # Тензор (batch_size=1)
    x = torch.tensor([indices], dtype=torch.long).to(device)
    
    # Предсказание
    with torch.no_grad():
        logits, _ = model(x)
        predictions = logits.argmax(dim=-1)
    
    # Декодируем
    chars = [idx_to_char.get(int(idx), '') for idx in predictions[0]]
    return ''.join(chars).rstrip()

In [68]:
text = "Hello World"
caesar_cipher(text=text, shift=3, alphabet=ENGLISH_ALPHABET)

'KhoorcZruog'

In [70]:
# Тест
encrypted = "KhoorcZruog".lower()
result = decrypt_with_model(model, encrypted, char_to_idx, idx_to_char, 
                            max_seq_len=50, device=device)
print(f"'{encrypted}' → '{result}'")

'khoorczruog' → 'hello world'


## Задание 2. Выполнить практическую работу из лекционного ноутбука

1. Построить RNN-ячейку на основе полносвязных слоев
1. Применить построенную ячейку для генерации текста с выражениями героев сериала "Симпсоны"